# VN-DARU — Chỉ số bất định pháp lý tài sản số cho Việt Nam

Notebook dựng chỉ số VN-DARU theo tuần từ GDELT (BigQuery) và kiểm định quan hệ với biến động thị trường Bitcoin.

- **Phần 1** — Thu thập dữ liệu và dựng bộ chỉ số (cần xác thực Google + gắn Drive + BigQuery).
- **Phần 2** — Phân tích và kiểm định (chỉ cần Drive, đọc file CSV do Phần 1 tạo).

Cài đặt gói trước khi chạy:

```
!pip install -q yfinance google-cloud-bigquery db-dtypes statsmodels
```

## Phần 1 — Thu thập dữ liệu và dựng chỉ số

In [ ]:
# ============================================================================
# VN-DARU — XÂY DỰNG CHỈ SỐ BẤT ĐỊNH PHÁP LÝ TÀI SẢN SỐ CHO VIỆT NAM
# Phần 1: Thu thập dữ liệu GDELT (BigQuery) và dựng bộ chỉ số theo tuần
# ----------------------------------------------------------------------------
# Yêu cầu môi trường: Google Colab (đã xác thực GCP + gắn Google Drive) và
# Internet cho Yahoo Finance.
# Cài gói trước khi chạy:
#     !pip install -q yfinance google-cloud-bigquery db-dtypes statsmodels
# ============================================================================

from google.colab import drive, auth
from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from dateutil.relativedelta import relativedelta
import os
import time
import warnings
warnings.filterwarnings("ignore")

# ---------------------------- CẤU HÌNH --------------------------------------
auth.authenticate_user()
drive.mount("/content/drive")

PROJECT_ID = "your-project-id"          # <-- thay bằng Project ID GCP của bạn
client = bigquery.Client(project=PROJECT_ID)

OUTPUT_PATH = "/content/drive/MyDrive/VN_DARU"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Mẫu tuần, bắt đầu và kết thúc canh đúng ngày Thứ Hai để mỗi tuần đủ 7 ngày.
START_DATE = datetime(2021, 1, 4)
END_DATE   = datetime(2026, 6, 29)

# Danh sách nguồn tin tiếng Việt dùng cho biến thể kiểm chứng theo domain.
VN_DOMAINS = [
    "vnexpress.net", "tuoitre.vn", "thanhnien.vn", "dantri.com.vn",
    "cafef.vn", "vietstock.vn", "vneconomy.vn", "baochinhphu.vn",
    "coin68.com", "tinnhanhchungkhoan.vn", "laodong.vn", "vietnamnet.vn",
    "nguoiquansat.vn", "thesaigontimes.vn", "vtv.vn", "vietnamplus.vn",
]

# Từ khóa tài sản số tiếng Việt, khớp trên phần path của URL. Ưu tiên dạng
# slug không dấu (phổ biến trong URL), giữ thêm một số dạng có dấu.
VN_KEYWORD_REGEX = (
    r"tai-san-so|tai-san-ma-hoa|tai-san-ao|tien-ma-hoa|tien-ao|"
    r"tien-ky-thuat-so|tien-so|tien-dien-tu|dong-tien-so|"
    r"tài-sản-số|tiền-điện-tử"
)

# ============================================================================
# TRUY VẤN CHÍNH — một lượt quét sinh 5 biến thể chỉ số song song
# ----------------------------------------------------------------------------
# Ba lớp điều kiện:
#   Lớp A (liên quan tài sản số): theme crypto/bitcoin/blockchain/defi/web3
#   Lớp B (liên quan pháp lý/bất định): họ theme EPU_* (theo logic Baker-Bloom
#          -Davis) cùng WB_678_TAXATION, LEGISLATION, ECON_REGULATION/
#          BANKRUPTCY/CENTRALBANK
#   Lớp C (liên quan Việt Nam): URL path chứa từ khóa tài sản số tiếng Việt
#
# 5 cột đếm (đều COUNT DISTINCT DocumentIdentifier trong phạm vi TUẦN):
#   n_crypto_global : Lớp A (tin crypto toàn cầu)
#   n_global_legal  : Lớp A + B
#   n_global_epu    : Lớp A + riêng EPU_*
#   n_vn_keyword    : Lớp A + B + từ khóa tiếng Việt trong URL  (CHỈ SỐ CHÍNH)
#   n_vn_domain     : Lớp A + B + nguồn thuộc danh sách domain VN
# ============================================================================

MAIN_SQL = """
WITH base AS (
  SELECT
    DATE_TRUNC(DATE(_PARTITIONTIME), WEEK(MONDAY)) AS week,
    DocumentIdentifier,
    SourceCommonName,
    (V2Themes LIKE '%EPU_%') AS is_epu,
    (
      V2Themes LIKE '%EPU_%'
      OR V2Themes LIKE '%WB_678_TAXATION%'
      OR V2Themes LIKE '%LEGISLATION%'
      OR REGEXP_CONTAINS(V2Themes, r'(^|;)ECON_(REGULATION|BANKRUPTCY|CENTRALBANK)')
    ) AS is_legal,
    REGEXP_CONTAINS(
      LOWER(REGEXP_EXTRACT(DocumentIdentifier, r'^([^?]+)')),
      r'{vn_kw}'
    ) AS has_vn_keyword,
    SourceCommonName IN UNNEST(@vn_domains) AS is_vn_domain
  FROM `gdelt-bq.gdeltv2.gkg_partitioned`
  WHERE _PARTITIONTIME >= TIMESTAMP('{start}')
    AND _PARTITIONTIME <  TIMESTAMP('{end}')
    AND (V2Themes LIKE '%CRYPTO%' OR V2Themes LIKE '%BITCOIN%' OR
         V2Themes LIKE '%BLOCKCHAIN%' OR V2Themes LIKE '%DEFI%' OR
         V2Themes LIKE '%WEB3%')
)
SELECT
  week,
  COUNT(DISTINCT DocumentIdentifier) AS n_crypto_global,
  COUNT(DISTINCT IF(is_legal, DocumentIdentifier, NULL)) AS n_global_legal,
  COUNT(DISTINCT IF(is_epu, DocumentIdentifier, NULL)) AS n_global_epu,
  COUNT(DISTINCT IF(is_legal AND has_vn_keyword, DocumentIdentifier, NULL)) AS n_vn_keyword,
  COUNT(DISTINCT IF(is_legal AND is_vn_domain, DocumentIdentifier, NULL)) AS n_vn_domain
FROM base
GROUP BY week
ORDER BY week
"""


def build_month_query(month_start: datetime, month_end: datetime):
    sql = MAIN_SQL.format(
        vn_kw=VN_KEYWORD_REGEX,
        start=month_start.strftime("%Y-%m-%d"),
        end=month_end.strftime("%Y-%m-%d"),
    )
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter("vn_domains", "STRING", VN_DOMAINS),
        ]
    )
    return sql, job_config


# Thu thập theo vòng lặp tháng để tránh vượt hạn mức BigQuery; mỗi tháng thử
# lại tối đa 3 lần, tháng thất bại được ghi lại để chạy bù (không điền 0).
MAX_RETRIES = 3
all_data = []
failed_months = []
current = START_DATE

print("Bắt đầu thu thập dữ liệu...")
while current < END_DATE:
    next_month = current + relativedelta(months=1)
    month_str = current.strftime("%Y-%m")
    sql, job_config = build_month_query(current, next_month)

    success = False
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            df_month = client.query(sql, job_config=job_config).to_dataframe()
            all_data.append(df_month)
            print(f"  {month_str}: OK ({len(df_month)} tuần)")
            success = True
            break
        except Exception as e:
            print(f"  {month_str}: lỗi lần {attempt}/{MAX_RETRIES} — {e}")
            time.sleep(5 * attempt)

    if not success:
        failed_months.append(month_str)
        print(f"  {month_str}: bỏ qua sau {MAX_RETRIES} lần thử — cần chạy bù riêng")

    current = next_month

if failed_months:
    print(f"\nCác tháng cần chạy bù: {failed_months}")
    with open(os.path.join(OUTPUT_PATH, "failed_months.txt"), "w") as f:
        f.write("\n".join(failed_months))

# ============================================================================
# TỔNG HỢP THEO TUẦN + DỰNG CHỈ SỐ
# ============================================================================
if not all_data:
    raise RuntimeError("Không thu thập được dữ liệu — kiểm tra lại cấu hình.")

raw = pd.concat(all_data, ignore_index=True)
raw["week"] = pd.to_datetime(raw["week"])

count_cols = [
    "n_crypto_global", "n_global_legal", "n_global_epu",
    "n_vn_keyword", "n_vn_domain",
]
weekly = raw.groupby("week")[count_cols].sum().reset_index()

# Điền đủ khung tuần liên tục và cắt về đúng biên [START_DATE, END_DATE].
full_weeks = pd.DataFrame({
    "week": pd.date_range(weekly["week"].min(), weekly["week"].max(), freq="7D")
})
weekly = full_weeks.merge(weekly, on="week", how="left").fillna(0)
weekly = weekly[
    (weekly["week"] >= pd.Timestamp(START_DATE))
    & (weekly["week"] < pd.Timestamp(END_DATE) - pd.Timedelta(days=6))
].reset_index(drop=True)

print(f"\nTổng số tuần thu được: {len(weekly)}")


def zscore_index(n: pd.Series) -> pd.Series:
    """Chỉ số chuẩn hóa: (N - trung bình) / độ lệch chuẩn + 100."""
    mu, sigma = n.mean(), n.std()
    return (n - mu) / sigma + 100


def epu_style_index(n: pd.Series) -> pd.Series:
    """Biến thể chuẩn hóa kiểu EPU (Baker et al., 2016): 100 * N / trung bình."""
    return 100 * n / n.mean()


for col in count_cols:
    suffix = col.replace("n_", "")
    weekly[f"idx_z_{suffix}"] = zscore_index(weekly[col])
    weekly[f"idx_epu_{suffix}"] = epu_style_index(weekly[col])

weekly["VN_DARU_main"]        = weekly["idx_z_vn_keyword"]
weekly["VN_DARU_global_only"] = weekly["idx_z_global_legal"]
weekly["VN_DARU_epu_only"]    = weekly["idx_z_global_epu"]
weekly["VN_DARU_vn_domain"]   = weekly["idx_z_vn_domain"]

weekly.to_csv(os.path.join(OUTPUT_PATH, "VN_DARU_weekly_all_variants.csv"), index=False)

# ============================================================================
# GHÉP BITCOIN REALIZED VOLATILITY (RV)
# ----------------------------------------------------------------------------
# RV là độ lệch chuẩn của log-return theo NGÀY trong tuần, đơn vị thập phân
# (ví dụ 0.03 tương ứng 3 điểm phần trăm/ngày).
# ============================================================================
import yfinance as yf

btc = yf.download("BTC-USD", start="2020-12-20", end="2026-07-02", progress=False)
if isinstance(btc.columns, pd.MultiIndex):
    btc.columns = btc.columns.get_level_values(0)
btc = btc[["Close"]].rename(columns={"Close": "close"})
btc["log_ret"] = np.log(btc["close"] / btc["close"].shift(1))
btc["week"] = btc.index.to_series().apply(
    lambda d: (d - pd.Timedelta(days=d.weekday())).normalize()
)
rv = (
    btc.dropna(subset=["log_ret"])
    .groupby("week")["log_ret"]
    .std(ddof=1)
    .rename("RV")
    .reset_index()
)

merged = weekly.merge(rv, on="week", how="inner")
merged.to_csv(os.path.join(OUTPUT_PATH, "VN_DARU_merged_with_RV.csv"), index=False)
print(f"Đã ghép RV — số tuần khớp: {len(merged)}")

# ============================================================================
# HÌNH TỔNG QUAN NHANH
# ============================================================================
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
axes[0].plot(weekly["week"], weekly["VN_DARU_main"], label="VN-DARU (main)")
axes[0].plot(weekly["week"], weekly["VN_DARU_global_only"], alpha=0.7, label="Global-legal")
axes[0].axhline(100, ls="--", lw=1, color="grey")
axes[0].set_title("VN-DARU vs. global-legal variant")
axes[0].legend(loc="upper left", fontsize=8); axes[0].grid(alpha=0.3)
axes[1].plot(weekly["week"], weekly["n_vn_keyword"])
axes[1].set_yscale("log"); axes[1].set_title("Raw weekly article counts — VN-keyword (log)")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, "VN_DARU_overview.png"), dpi=200)
plt.show()

print("\nHoàn tất Phần 1. Dữ liệu đã lưu:")
print(" -", os.path.join(OUTPUT_PATH, "VN_DARU_weekly_all_variants.csv"))
print(" -", os.path.join(OUTPUT_PATH, "VN_DARU_merged_with_RV.csv"))


## Phần 2 — Phân tích và kiểm định

In [ ]:
# ============================================================================
# VN-DARU — Phần 2: Phân tích và kiểm định
# ----------------------------------------------------------------------------
# Đọc VN_DARU_merged_with_RV.csv (do Phần 1 tạo). Chỉ cần Google Drive.
# ============================================================================
import pandas as pd, numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.tsa.stattools import grangercausalitytests, adfuller, kpss
from scipy import stats as st
import matplotlib.pyplot as plt, matplotlib.dates as mdates
import warnings; warnings.filterwarnings("ignore")

PATH = "/content/drive/MyDrive/VN_DARU/VN_DARU_merged_with_RV.csv"
COUNT = ["n_crypto_global", "n_global_legal", "n_global_epu",
         "n_vn_keyword", "n_vn_domain"]

# --- Chuẩn bị dữ liệu -------------------------------------------------------
# GDELT không phát sinh bản ghi GKG trong hai tuần 16/6 và 23/6/2025 (gián
# đoạn hạ tầng), khiến toàn bộ biến — kể cả tin crypto toàn cầu vốn ~30-40
# nghìn bài/tuần — rơi về 0; các tuần liền kề chỉ được bao phủ một phần. Vì
# đây là khuyết dữ liệu chứ không phải môi trường tin tức, khoảng 02/6-30/6/
# 2025 được coi là khuyết và loại khỏi phân tích. Khoảng loại này đồng thời
# tạo "donut window" quanh ngày ban hành Luật 71 (14/6) và Nghị quyết 222
# (27/6). Chỉ số được chuẩn hóa trên mẫu hợp lệ còn lại.
OUTAGE_START, OUTAGE_END = pd.Timestamp("2025-06-02"), pd.Timestamp("2025-06-30")
LAW_PRE_END, LAW_POST_START = OUTAGE_START, OUTAGE_END

m = pd.read_csv(PATH); m["week"] = pd.to_datetime(m["week"])
bad = (m["week"] >= OUTAGE_START) & (m["week"] <= OUTAGE_END)
d = m.loc[~bad].reset_index(drop=True).copy()
print(f"Loại {int(bad.sum())} tuần khuyết dữ liệu ({m.loc[bad,'week'].dt.date.tolist()})")
print(f"Mẫu hợp lệ: N = {len(d)} tuần")

for c in COUNT:
    s = d[c].astype(float)
    d["z_" + c[2:]] = (s - s.mean()) / s.std(ddof=1) + 100
d["DARU"] = d["z_vn_keyword"]; d["GLOB"] = d["z_global_legal"]
d["EPUv"] = d["z_global_epu"]; d["DOM"] = d["z_vn_domain"]; d["CG"] = d["z_crypto_global"]

# --- Bảng 3: Thống kê mô tả -------------------------------------------------
nk = d["n_vn_keyword"]
print("\n=== Bảng 3 — Thống kê mô tả ===")
print(f"VN-keyword N_t : mean={nk.mean():.1f} sd={nk.std(ddof=1):.1f} min={nk.min():.0f} "
      f"max={nk.max():.0f} skew={st.skew(nk):.2f} CV={nk.std(ddof=1)/nk.mean()*100:.0f}% "
      f"zeros={(nk==0).sum()}")
print(f"VN-DARU index  : mean={d['DARU'].mean():.2f} sd={d['DARU'].std(ddof=1):.2f} "
      f"min={d['DARU'].min():.2f} max={d['DARU'].max():.2f}")
print(f"VN-domain      : mean={d['n_vn_domain'].mean():.1f} sd={d['n_vn_domain'].std(ddof=1):.1f}")
print(f"Global-legal   : mean={d['n_global_legal'].mean():.0f} sd={d['n_global_legal'].std(ddof=1):.0f}")
print(f"RV             : mean={d['RV'].mean():.4f} sd={d['RV'].std(ddof=1):.4f} "
      f"min={d['RV'].min():.4f} max={d['RV'].max():.4f}")

# --- Bảng 4: Ma trận tương quan (kiểm chứng giá trị) ------------------------
print("\n=== Bảng 4 — Ma trận tương quan (chuẩn hóa) ===")
cm = d[["CG", "GLOB", "EPUv", "DARU", "DOM"]].corr()
cm.columns = cm.index = ["Crypto-global", "Legal-global", "EPU-only", "VN-keyword", "VN-domain"]
print(cm.round(3).to_string())
rc = st.pearsonr(d["DARU"], d["DOM"]); rd = st.pearsonr(d["DARU"], d["GLOB"])
print(f"Hội tụ   VN-keyword ~ VN-domain : r={rc[0]:.3f} (p={rc[1]:.1e})")
print(f"Phân biệt VN-keyword ~ global   : r={rd[0]:.3f} (p={rd[1]:.3f})")

# --- Bảng 5: 10 tuần cao nhất -----------------------------------------------
print("\n=== Bảng 5 — 10 tuần cao nhất theo VN-DARU ===")
print(d.nlargest(10, "DARU")[["week", "n_vn_keyword", "DARU", "n_vn_domain"]].to_string(index=False))

# --- Tính dừng + AR(1) ------------------------------------------------------
print("\n=== Kiểm định tính dừng ===")
for nm in ["DARU", "RV", "GLOB"]:
    adf_s, adf_p = adfuller(d[nm].dropna(), autolag="AIC")[:2]
    try:    kp_s, kp_p = kpss(d[nm].dropna(), regression="c", nlags="auto")[:2]
    except: kp_s, kp_p = np.nan, np.nan
    print(f"{nm:5s}: ADF stat={adf_s:.2f} (p={adf_p:.3f}) | KPSS stat={kp_s:.2f} (p={kp_p})")
print(f"AR(1) VN-DARU = {d['DARU'].autocorr(1):.3f}")

# --- Bảng 6: H2 (VN-DARU ~ RV) ---------------------------------------------
print("\n=== Bảng 6 — H2: VN-DARU ~ RV ===")
HAC = {"cov_type": "HAC", "cov_kwds": {"maxlags": 5}}
m_ols = smf.ols("DARU ~ RV", d).fit()
m_hac = smf.ols("DARU ~ RV", d).fit(**HAC)
print(f"Baseline OLS : b_RV={m_ols.params['RV']:.2f} se={m_ols.bse['RV']:.2f} "
      f"p={m_ols.pvalues['RV']:.1e} R2={m_ols.rsquared*100:.1f}% "
      f"DW={sm.stats.durbin_watson(m_ols.resid):.2f}")
print(f"HAC (lag5)   : b_RV={m_hac.params['RV']:.2f} se={m_hac.bse['RV']:.2f} p={m_hac.pvalues['RV']:.3f}")
d["DARU_lag1"] = d["DARU"].shift(1); d["RV_lag1"] = d["RV"].shift(1)
m_lag = smf.ols("DARU ~ RV + DARU_lag1", d.dropna(subset=["DARU_lag1"])).fit()
print(f"+lag DV      : b_RV={m_lag.params['RV']:.2f} p={m_lag.pvalues['RV']:.1e} "
      f"b_lag={m_lag.params['DARU_lag1']:.2f} R2={m_lag.rsquared*100:.1f}%")
m_rl = smf.ols("DARU ~ RV_lag1", d.dropna(subset=["RV_lag1"])).fit()
print(f"RV lag1      : b={m_rl.params['RV_lag1']:.2f} p={m_rl.pvalues['RV_lag1']:.1e} R2={m_rl.rsquared*100:.1f}%")

# --- Bảng 7 & 8: H3 ---------------------------------------------------------
pre  = d[d["week"] <  LAW_PRE_END]
post = d[d["week"] >  LAW_POST_START]
print(f"\n=== Bảng 7 — H3 dạng mức (pre N={len(pre)}, post N={len(post)}) ===")
tt = st.ttest_ind(pre["DARU"], post["DARU"], equal_var=False)
mw = st.mannwhitneyu(pre["DARU"], post["DARU"])
sp = np.sqrt(((len(pre)-1)*pre["DARU"].std()**2 + (len(post)-1)*post["DARU"].std()**2)
             / (len(pre)+len(post)-2))
dcoh = (pre["DARU"].mean() - post["DARU"].mean()) / sp
print(f"pre  mean={pre['DARU'].mean():.2f} sd={pre['DARU'].std():.2f} | "
      f"post mean={post['DARU'].mean():.2f} sd={post['DARU'].std():.2f}")
print(f"diff={post['DARU'].mean()-pre['DARU'].mean():.3f} | Welch p={tt.pvalue:.3f} | "
      f"MW p={mw.pvalue:.3f} | Cohen d={dcoh:.3f}")

print("\n=== Bảng 8 — H3 dạng độ nhạy ===")
d["post"] = (d["week"] > LAW_POST_START).astype(int)
m_pr  = smf.ols("DARU ~ post + RV", d).fit(**HAC)
m_int = smf.ols("DARU ~ post + RV + post:RV", d).fit(**HAC)
print(f"post+RV   : post={m_pr.params['post']:.2f}(p={m_pr.pvalues['post']:.2f}) "
      f"RV={m_pr.params['RV']:.2f}(p={m_pr.pvalues['RV']:.3f}) R2={m_pr.rsquared*100:.1f}%")
print(f"tương tác : post={m_int.params['post']:.2f}(p={m_int.pvalues['post']:.3f}) "
      f"RV={m_int.params['RV']:.2f}(p={m_int.pvalues['RV']:.3f}) "
      f"post:RV={m_int.params['post:RV']:.2f}(p={m_int.pvalues['post:RV']:.3f}) R2={m_int.rsquared*100:.1f}%")
m_pre  = smf.ols("DARU ~ RV", pre).fit(**HAC)
m_post = smf.ols("DARU ~ RV", post).fit(**HAC)
print(f"pre-law RV ={m_pre.params['RV']:.2f} (p={m_pre.pvalues['RV']:.3f}) | "
      f"post-law RV={m_post.params['RV']:.2f} (p={m_post.pvalues['RV']:.3f})")
for yv, lab in [("GLOB", "global-legal"), ("DOM", "VN-domain")]:
    mc = smf.ols(f"{yv} ~ post", d).fit(**HAC)
    print(f"Đối chứng {lab:12s}: post={mc.params['post']:.2f} (p={mc.pvalues['post']:.3f})")

# --- Placebo split ----------------------------------------------------------
print("\n=== Placebo split (mô hình tương tác) ===")
for sd in ["2022-06-14", "2023-06-14", "2024-06-14", "2025-06-14"]:
    tmp = d.copy(); tmp["post"] = (tmp["week"] > pd.Timestamp(sd)).astype(int)
    mm = smf.ols("DARU ~ post + RV + post:RV", tmp).fit(**HAC)
    print(f"{sd}: post={mm.params['post']:.2f}(p={mm.pvalues['post']:.3f}) "
          f"post:RV={mm.params['post:RV']:.1f}(p={mm.pvalues['post:RV']:.3f})")

# --- Granger hai chiều ------------------------------------------------------
print("\n=== Granger hai chiều ===")
gc = d[["DARU", "RV"]].dropna().reset_index(drop=True)
for a, b in [("DARU", "RV"), ("RV", "DARU")]:
    print(f"-- {b} -> {a}:")
    res = grangercausalitytests(gc[[a, b]], maxlag=4, verbose=False)
    for lag in range(1, 5):
        print(f"   lag{lag}: p={res[lag][0]['ssr_ftest'][1]:.4f}")

# --- Bảng 9: theo pha -------------------------------------------------------
print("\n=== Bảng 9 — VN-DARU theo pha ===")
for nm, s, e in [("Gray area", "2021-01-04", "2025-06-01"),
                 ("Law71+Res222 (Jul-Sep 25)", "2025-07-07", "2025-09-07"),
                 ("Res05 pilot (Sep25-Jan26)", "2025-09-08", "2026-01-18"),
                 ("Licensing (from 20/01/26)", "2026-01-19", "2026-06-22")]:
    seg = d[(d["week"] >= s) & (d["week"] <= e)]["DARU"]
    print(f"  {nm:28s} N={len(seg):3d} mean={seg.mean():.2f} sd={seg.std(ddof=1):.2f}")

# --- Hình 1 & 2 -------------------------------------------------------------
mil = [("2025-06-14", "Law 71/2025"), ("2025-06-27", "Res 222/2025"),
       ("2025-09-09", "Res 05/2025"), ("2026-01-20", "MoF licensing")]
mu, sg = d["n_vn_keyword"].mean(), d["n_vn_keyword"].std(ddof=1)
gmu, gsg = d["n_global_legal"].mean(), d["n_global_legal"].std(ddof=1)
full = m.copy()
full["DARU_p"] = ((full["n_vn_keyword"].astype(float).mask(bad)) - mu) / sg + 100
full["GLOB_p"] = ((full["n_global_legal"].astype(float).mask(bad)) - gmu) / gsg + 100

fig, ax = plt.subplots(2, 1, figsize=(11, 7.6), sharex=True)
ax[0].plot(full["week"], full["n_vn_keyword"].mask(bad), color="#1f4e79", lw=1.1)
ax[0].set_yscale("log"); ax[0].set_ylabel("Articles / week (log)")
ax[0].set_title("Panel A. Raw weekly article counts — VN-keyword (Layers A+B+C)", fontsize=10, loc="left")
ax[1].axhline(100, color="0.6", ls="--", lw=.8)
ax[1].plot(full["week"], full["DARU_p"], color="#1f4e79", lw=1.2, label="VN-DARU (main)")
ax[1].plot(full["week"], full["GLOB_p"], color="#2ca02c", lw=1.0, alpha=.8, label="Global-legal control")
ax[1].set_ylabel("Index (mean=100)"); ax[1].legend(fontsize=8, loc="upper left")
ax[1].set_title("Panel B. Standardized VN-DARU vs. global-legal control", fontsize=10, loc="left")
for a_ in ax:
    a_.axvspan(OUTAGE_START, OUTAGE_END, color="0.85")
    for dt, _ in mil:
        a_.axvline(pd.Timestamp(dt), color="#c0392b", ls=":", lw=1.0)
ax[1].xaxis.set_major_locator(mdates.YearLocator())
ax[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout(); plt.savefig("fig1.png", dpi=300, bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(figsize=(7.2, 5))
ax.scatter(pre["RV"], pre["DARU"], s=18, color="#c0392b", alpha=.55, label="Pre-Law 71")
ax.scatter(post["RV"], post["DARU"], s=26, color="#1f4e79", alpha=.8, label="Post-Law 71")
xs = np.linspace(d["RV"].min(), d["RV"].max(), 50)
bp = np.polyfit(pre["RV"], pre["DARU"], 1); bq = np.polyfit(post["RV"], post["DARU"], 1)
ax.plot(xs, np.polyval(bp, xs), color="#c0392b", lw=2, label=f"Pre slope beta={bp[0]:.1f}")
ax.plot(xs, np.polyval(bq, xs), color="#1f4e79", lw=2, label=f"Post slope beta={bq[0]:.1f}")
ax.set_xlabel("Bitcoin realized volatility (weekly)"); ax.set_ylabel("VN-DARU (mean=100)")
ax.set_title("Figure 2. VN-DARU vs. Bitcoin volatility, pre/post Law 71/2025", fontsize=10, loc="left")
ax.legend(fontsize=8); plt.tight_layout(); plt.savefig("fig2.png", dpi=300, bbox_inches="tight"); plt.show()

print("\nHoàn tất Phần 2. Đã lưu fig1.png, fig2.png.")
